# TV-FLIDS - Full Experimental Campaign (Colab, T4)

Run this notebook **top to bottom**. Each section is independent enough to
re-run on its own after a disconnect.

## What this notebook is

An **orchestration layer**. It does not reimplement any part of the science:
every experiment is executed by the repository's own runners under
`experiments/`, and every table and figure is produced by the repository's own
generators under `scripts/`. This notebook clones the code, pins the commit,
installs the pinned stack, attaches durable storage, measures the right
concurrency, runs a bounded parallel queue, and validates the result.

## What this notebook will not do

It will not invent a number. There is no expected accuracy, F1, ASR, p-value or
improvement anywhere in `expected_results/` - that directory is a contract for
**structure and completeness only**. The manuscript's existing values are
**not** targets: any historical figure lives under
`expected_results/historical_reference/`, is labelled
**HISTORICAL - NON-TARGET**, and is refused as a comparison source by the
validator.

The only legitimate final numbers are the ones these runs produce, recorded
with full provenance under `campaign_results/`.

## The campaign

**580 executable cells**, 100 rounds each, across 12 phases. One phase
(`K_ciciot2023`, Supplementary Table S4 / Figure S1) is **blocked**: the
CIC-IoT-2023 dataset is not in the repository. It is enumerated and reported as
blocked, never faked.

| Priority | Phases | Paper artifacts |
| --- | --- | --- |
| 1 - main paper | A, C, D, N, E, F, R | Table V, Figures 2-4, VII, VIII, IX-X, XI, Figure 5 |
| 2 - supporting | B, G (+ L derived) | Table VI, Section VIII-B, Table XII |
| 3 - supplementary | H, I, J | Tables S1, S2, S3 |

Priority order matters: if the session dies early, the highest-value scientific
artifacts are the ones most likely to already exist.

---
# Section 1 - Environment

Reports the runtime and **fails early** if it is not compatible. Run this
before anything else.

In [ ]:
#@title 1.1 - Raw runtime facts (before any install)
import subprocess, sys, platform, os, shutil

print("Python           ", platform.python_version(), "|", sys.executable)
print("Platform         ", platform.system(), platform.release(), platform.machine())
print("CPU count        ", os.cpu_count())

try:
    with open("/proc/meminfo") as fh:
        for line in fh:
            if line.startswith(("MemTotal", "MemAvailable")):
                k, v = line.split(":")
                print(f"{k:<17}", round(int(v.split()[0]) / 1024**2, 2), "GB")
except OSError:
    pass

for label, path in (("/content", "/content"), ("/", "/")):
    if os.path.isdir(path):
        du = shutil.disk_usage(path)
        print(f"disk {label:<12}", round(du.free/1024**3, 1), "GB free of",
              round(du.total/1024**3, 1), "GB")

print()
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi absent - no GPU is attached to this runtime.")
    print("Runtime > Change runtime type > Hardware accelerator: T4 GPU")

**Stop here if the cell above shows no T4.** The campaign will otherwise run
on CPU, at roughly the throughput this migration was meant to escape: the
previous CPU host measured **about 21.5 min per 100-round cell**.

---
# Section 2 - Repository

Prefer **Option A (clone)**: a clone records a commit, so every result can name
the code that produced it. Option B (upload) is a fallback and records no
commit unless the archive contains `.git`.

In [ ]:
#@title 2.1 - Option A: clone and pin the exact commit  (PREFERRED)
REPO_URL = "https://github.com/aliakarma/TV-FLIDS.git"  #@param {type:"string"}
BRANCH   = "main"                                        #@param {type:"string"}
# Pin the commit the campaign runs. Empty = branch HEAD (NOT reproducible).
COMMIT   = ""                                             #@param {type:"string"}
CLONE_PATH = "/content/TV-FLIDS"                         #@param {type:"string"}

import os, subprocess, sys

def sh(*a, **kw):
    print("$", " ".join(a))
    return subprocess.run(a, check=kw.pop("check", True), **kw)

if not os.path.isdir(os.path.join(CLONE_PATH, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, CLONE_PATH)
else:
    print(f"{CLONE_PATH} already contains a clone; fetching")
    sh("git", "-C", CLONE_PATH, "fetch", "--all", "--tags")

if COMMIT.strip():
    sh("git", "-C", CLONE_PATH, "checkout", "--detach", COMMIT.strip())
else:
    print("\n  WARNING: no COMMIT pinned. The campaign will run whatever the")
    print("  branch happens to point at, which is not reproducible.\n")

os.chdir(CLONE_PATH)
sys.path.insert(0, CLONE_PATH)
print("\ncwd:", os.getcwd())

In [ ]:
#@title 2.2 - Option B: upload an archive  (FALLBACK - run only if 2.1 failed)
# Leave this unexecuted when the clone above succeeded.
RUN_UPLOAD = False  #@param {type:"boolean"}

if RUN_UPLOAD:
    import os, sys, tarfile, zipfile, shutil
    from google.colab import files
    up = files.upload()                     # pick TV-FLIDS.zip or .tar.gz
    name = next(iter(up))
    dest = "/content/TV-FLIDS"
    shutil.rmtree(dest, ignore_errors=True)
    os.makedirs(dest, exist_ok=True)
    if name.endswith((".tar.gz", ".tgz", ".tar")):
        with tarfile.open(name) as tf:
            tf.extractall("/content")
    elif name.endswith(".zip"):
        with zipfile.ZipFile(name) as zf:
            zf.extractall("/content")
    else:
        raise SystemExit(f"unsupported archive: {name}")
    # An archive may unpack into a single top-level directory.
    if not os.path.exists(os.path.join(dest, "experiments")):
        for e in sorted(os.listdir("/content")):
            p = os.path.join("/content", e)
            if os.path.isdir(p) and os.path.exists(os.path.join(p, "experiments")):
                dest = p
                break
    os.chdir(dest)
    sys.path.insert(0, dest)
    print("cwd:", os.getcwd())
    print("\n  NOTE: an upload records no commit unless the archive carried")
    print("  .git. Every result's provenance will say the commit is unknown.")
else:
    print("skipped (clone path used)")

In [ ]:
#@title 2.3 - Record exactly what code this campaign runs
import subprocess

def git(*a):
    try:
        return subprocess.check_output(["git", *a], text=True,
                                        stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

state = {
    "remote":   git("config", "--get", "remote.origin.url"),
    "commit":   git("rev-parse", "HEAD"),
    "branch":   git("rev-parse", "--abbrev-ref", "HEAD"),
    "describe": git("describe", "--always", "--dirty"),
}
status = git("status", "--porcelain") or ""
state["dirty"] = bool(status.strip())
state["dirty_paths"] = [l[3:] for l in status.splitlines() if l.strip()]

for k in ("remote", "commit", "branch", "describe", "dirty"):
    print(f"  {k:<12} {state[k]}")
print(f"  {'dirty_paths':<12} {len(state['dirty_paths'])}")
for p in state["dirty_paths"][:10]:
    print(f"               {p}")

if state["dirty"]:
    print("\n  A dirty tree is recorded in every result's provenance. That is")
    print("  honest, but the commit alone then does not reproduce the run.")
if not state["commit"]:
    print("\n  No commit available - provenance will say so explicitly.")

---
# Section 3 - Dependencies

Installs the versions `requirements.txt` pins. Nothing is silently upgraded,
and the final report prints what actually landed so a difference is **visible**
rather than assumed.

`torch` comes from the CUDA 12.1 wheel index: the `+cpu` build cannot use the
T4 however healthy `nvidia-smi` looks. Note that `2.1.0+cu121` **is** torch
2.1.0 - the suffix names the build variant, not the release.

In [ ]:
#@title 3.1 - Install the pinned stack (a few minutes)
!bash colab/setup_colab.sh

### If the cell above warned about Python

Colab's default Python is usually newer than the 3.10 these wheels target. The
campaign still runs, and every result records the **actual** versions - but do
not then describe the environment as matching the paper's stack. That claim
requires the versions to actually match, which the next cell reports.

In [ ]:
#@title 3.2 - The repository's own environment report (reports, never asserts)
!python scripts/verify_environment.py

In [ ]:
#@title 3.3 - Hard compatibility gate  (fails early, on purpose)
# A non-zero exit here means: do not start the campaign yet.
!python colab/check_colab_environment.py --require-gpu \
        --json results/_campaign/env_report.json ; echo "exit=$?" 

---
# Section 4 - Persistent storage (Google Drive)

A Colab session can end at any time. Mount Drive so the campaign survives it.

**How the mapping works.** The repository's runners and its table generator
both hardcode a `results/...` prefix. Rather than fork that path handling,
`results` becomes a **symlink** to the campaign root on Drive. The runners
write `results/logs/comparison/...` unchanged; the bytes land at
`campaign_results/logs/comparison/...`, which is exactly the relative path
`expected_results/` mirrors:

```
expected_results/logs/comparison/fedavg_label_flip_30_seed42/experiment_log.json   <- contract
campaign_results/logs/comparison/fedavg_label_flip_30_seed42/experiment_log.json   <- result
```

In [ ]:
#@title 4.1 - Mount Drive
MOUNT_DRIVE = True  #@param {type:"boolean"}

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !ls -la /content/drive/MyDrive | head -5
else:
    print("Drive not mounted. Storage will be SESSION-LOCAL and LOST on")
    print("disconnect. Only do this for a throwaway test.")

In [ ]:
#@title 4.2 - Point results/ at the durable campaign root
# Refuses to replace a non-empty real results/ directory; it tells you how to
# merge instead. Add --local to force session-local storage.
!python colab/run_campaign.py link

In [ ]:
#@title 4.3 - Confirm the mapping is live
import os
print("results ->", os.readlink("results") if os.path.islink("results")
      else "NOT A SYMLINK (session-local or unlinked)")
print()
!ls -la results/
print()
!df -h results/ | tail -2

---
# Section 5 - Campaign inventory and the expected-results contract

`colab/campaign_inventory.py` is the single authoritative enumeration of every
cell. The queue, the `expected_results/` tree and the validator all derive from
it, so expected and actual paths are isomorphic **by construction** - there is
no second place where a path could be spelled differently.

The `verify` step below imports the repository's real runner constants and
asserts every grid matches. A mismatch is a hard error, not a warning.

In [ ]:
#@title 5.1 - The inventory, and the drift guard against the repo
!python colab/campaign_inventory.py summary
print()
!python colab/campaign_inventory.py verify

In [ ]:
#@title 5.2 - (Re)generate the expected_results/ contract tree
# 598 files: 580 per-cell contracts + 12 aggregate contracts + the manifest
# template, paper-artifact index, figure/statistics contracts and the
# HISTORICAL-NON-TARGET holder. Contains no expected numerical outcome.
!python colab/generate_expected_results.py
print()
!ls expected_results/
print()
!python -c "import json;b=json.load(open('expected_results/manifest_template.json'));print('template cells:',b['n_cells']);print(json.dumps(b['cells_by_phase'],indent=2))"

In [ ]:
#@title 5.3 - Prove expected/actual isomorphism, and that no target was invented
# Assertion 1: every declared result path has a contract.
# Assertion 2: every contract corresponds to a declared path (no stale ones).
# Assertion 3: no result sits at a path the contract does not describe.
# Assertion 4: no contract carries a predicted accuracy/F1/ASR/p-value/delta.
!python colab/check_isomorphism.py --results-root results --scaffold

---
# Section 6 - Pre-campaign smoke test

**Do not skip this.** One deliberately tiny TV-FLIDS run (few clients, few
rounds, one seed, one attack) into a throwaway path, then eleven assertions on
the artifact it produced:

1. the run completed and wrote `experiment_log.json`
2. the round log has `rounds+1` entries indexed `0..rounds`
3. the model actually trained
4. the attack executed (malicious clients selected and recorded)
5. the verification gate executed
6. the trust EMA executed
7. the STE / meta-gradient executed (alpha+beta+gamma = 1 every round)
8. provenance is complete and the config hash is real (never `mockhash`)
9. **the GPU was genuinely used** (binding whenever CUDA is available)
10. the artifact matches the `expected_results/` schema and metric ranges
11. no mock or quarantined artifact was consumed

Every assertion reads the artifact. None checks a metric against an expected
value - a smoke test proves the machinery ran, not what it should conclude.

In [ ]:
#@title 6.1 - Run the smoke test
!python colab/run_campaign.py smoke

**If any assertion says FAIL, stop and fix it.** Spending the campaign on a
broken pipeline produces 580 worthless cells. The transcript is at
`results/_campaign_logs/colab_smoke/smoke_run.log`.

---
# Section 7 - Concurrency benchmark

One T4 is one GPU. Two concurrent workers can **help** (each hides the other's
Ray/Flower start-up and per-round Python overhead) or **hurt** (they contend
for the same SMs, and two 20-client simulations can exhaust GPU memory or host
RAM). Which one happens is a property of this runtime, so it is **measured**.

The default stays conservative: `MAX_PARALLEL_JOBS = 1` unless the measurement
says otherwise. A mode with **any** failure is never recommended, however fast
it looked - the objective is maximum *stable* completed cells per hour.

In [ ]:
#@title 7.1 - Compare 1 vs 2 concurrent workers (short jobs)
!python colab/run_campaign.py benchmark

In [ ]:
#@title 7.2 - Adopt the measured concurrency
# Set these to what the benchmark recommended, then run this cell.
MAX_PARALLEL_JOBS = 1   #@param {type:"integer"}
SIM_CLIENT_GPUS   = 0.1 #@param {type:"number"}
SIM_CLIENT_CPUS   = 1   #@param {type:"integer"}

import re, io
p = "colab/config_colab.yaml"
s = io.open(p, encoding="utf-8").read()
s = re.sub(r"max_parallel_jobs:\s*\S+", f"max_parallel_jobs: {MAX_PARALLEL_JOBS}", s)
s = re.sub(r"sim_client_gpus:\s*\S+",   f"sim_client_gpus: {SIM_CLIENT_GPUS}", s)
s = re.sub(r"sim_client_cpus:\s*\S+",   f"sim_client_cpus: {SIM_CLIENT_CPUS}", s)
io.open(p, "w", encoding="utf-8").write(s)

!grep -A8 "^execution:" colab/config_colab.yaml

---
# Section 8 - Run the campaign

## How the queue works

`colab/parallel_runner.py` maintains a persistent queue at
`results/_campaign/colab/queue.json`, written atomically after every state
change, with statuses `pending / running / complete / failed / retrying`.

**Two stages, because the repository's runners come in two shapes.**

- **Stage 1 - cell production (parallel, bounded).** For phases where a cell is
  a plain `run_experiment.py` invocation (A, B, D, E, F, G, J, N), each cell is
  its own job: maximum parallelism, and a crash loses at most one cell. For
  phases whose runner writes a temp YAML per cell (C ablation, H/I
  hyperparameter grids, R ratio sweep), those configs are *scientific content*,
  so the runner refuses to reproduce them - it **shards the family runner**
  instead, along the axes the CLI already exposes, each shard writing its
  aggregate to a throwaway directory.
- **Stage 2 - aggregation (sequential, cheap).** Each phase's family runner is
  invoked once over the **full** grid with `TVFLIDS_RESUME=1` and the real
  `--output`. Every cell is already on disk, so the runner reuses each stored
  summary and does only what it alone should do: aggregate, run the statistics,
  and write the phase artifact. A phase whose cells are not all complete is
  **skipped**, so no partial artifact is ever written.

## Resumability

`TVFLIDS_RESUME=1` means a cell with a complete `experiment_log.json` returns
its own stored summary instead of recomputing. So after a disconnect: **just
re-run the cell below.** Completed cells are skipped, `running` units are
re-queued (their process is gone), and failed units are retried up to
`max_attempts`. No completed result is ever overwritten.

In [ ]:
#@title 8.1 - Dry run: see exactly what will execute
!python colab/run_campaign.py run --dry-run 2>&1 | head -30

In [ ]:
#@title 8.2 - Priority 1 - MAIN PAPER (Table V, Figures 2-4, VII, VIII, IX-X, XI, Figure 5)
# Safe to re-run after a disconnect: completed cells are skipped.
!python colab/run_campaign.py run --priority 1

In [ ]:
#@title 8.3 - Priority 2 - supporting analysis (Table VI, Section VIII-B)
!python colab/run_campaign.py run --priority 2

In [ ]:
#@title 8.4 - Priority 3 - supplementary (Tables S1, S2, S3)
!python colab/run_campaign.py run --priority 3

In [ ]:
#@title 8.5 - Queue snapshot (safe to run any time)
!python colab/run_campaign.py status 2>&1 | head -40

In [ ]:
#@title 8.6 - Retry whatever failed
!python colab/run_campaign.py run --retry-failed

---
# Section 9 - Finalize and validate

`finalize` writes `campaign_results/final_manifest.json` - an index of what
genuinely exists, cell by cell, with each cell's own recorded metrics, config
hash, git commit and log SHA-256. A cell that never ran is `pending`; a
truncated one is `failed` with the reason. Nothing is invented.

`validate` then compares expected against actual:

| Group | Checks |
| --- | --- |
| completeness | every cell, seed, round and output file exists |
| integrity | no `mockhash`, no quarantined output at a live path, no zero-byte artifact, no malformed PDF, no contract file masquerading as a result |
| provenance | git commit, config hash, seed, dataset, environment |
| numerical | every metric inside its mathematically valid range; alpha+beta+gamma = 1 |
| statistical | p-values are probabilities, no negative sigma, seed counts, ASR framed lower-is-better |
| manuscript | every table and figure has a live source; nothing still provisional |
| manifest | template vs. `final_manifest.json`, and every "complete" claim verified on disk |

In [ ]:
#@title 9.1 - Write final_manifest.json
!python colab/run_campaign.py finalize

In [ ]:
#@title 9.2 - Validate the campaign
!python colab/run_campaign.py validate

In [ ]:
#@title 9.3 - Per-seed result map (match every result to its paper cell)
# Experiment | Seed | Status | Config Hash | Rounds | Output
!python colab/run_campaign.py report 2>&1 | tail -80

---
# Section 10 - Tables and figures

**Only run this once a phase family is complete and validated.** Do not edit
the manuscript as individual cells finish: finish a family, validate every
cell, aggregate, validate the statistics, and only then regenerate that
family's tables and figures. Partial regeneration produces an inconsistent
manuscript.

Both generators **skip** anything whose backing artifact is absent rather than
inventing it, so running this early is safe - it simply produces less. The
`--check` pass reports what is still unbacked.

In [ ]:
#@title 10.1 - Regenerate manuscript tables and figure bodies from real artifacts
!python colab/run_campaign.py artifacts

In [ ]:
#@title 10.2 - The repository's own result and manuscript integrity checks
!python scripts/check_results.py    ; echo "check_results exit=$?"
!python scripts/check_manuscript.py ; echo "check_manuscript exit=$?" 

---
# Section 11 - Archive

Copy the campaign off the session. If `results` is already a Drive symlink,
everything is on Drive as it is written and this is only a convenience
snapshot.

In [ ]:
#@title 11.1 - Snapshot the campaign to Drive
import os, subprocess, datetime
DEST = "/content/drive/MyDrive/TV-FLIDS-Campaign"  #@param {type:"string"}

stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DEST, exist_ok=True)
    # The contract tree and the orchestration code travel with the results.
    for src in ("expected_results", "colab"):
        subprocess.run(["cp", "-a", src, os.path.join(DEST, f"{src}_{stamp}")])
    if not os.path.islink("results"):
        subprocess.run(["cp", "-a", "results",
                        os.path.join(DEST, f"campaign_results_{stamp}")])
    else:
        print("results is a Drive symlink - the results are already on Drive.")
    print("archived to", DEST)
else:
    print("Drive is not mounted; nothing archived.")

---
# Appendix - Minimal sequence

```bash
# once per session
bash colab/setup_colab.sh
python colab/check_colab_environment.py --require-gpu
python colab/run_campaign.py link

# once, before committing to the campaign
python colab/check_isomorphism.py --results-root results --scaffold
python colab/run_campaign.py smoke
python colab/run_campaign.py benchmark      # then set max_parallel_jobs

# the campaign (re-run any of these after a disconnect)
python colab/run_campaign.py run --priority 1
python colab/run_campaign.py run --priority 2
python colab/run_campaign.py run --priority 3

# after each phase family completes
python colab/run_campaign.py finalize
python colab/run_campaign.py validate
python colab/run_campaign.py artifacts
```

## The workflow this enforces

```
Colab notebook -> real experiment -> seed-specific output -> real provenance
   -> validated result -> table/figure -> paper
```

and never

```
paper number -> expected number -> modified code -> fake result
```

`expected_results/` is a contract for **structure and completeness**. It is not
a source of numerical truth, and it contains none.